In [ ]:
%pip install -U transformers accelerate


## Local Inference on GPU 
Model page: https://huggingface.co/zai-org/GLM-OCR

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/zai-org/GLM-OCR)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
import gc
import torch
from transformers import pipeline

model_path = "zai-org/GLM-OCR"
messages = [{"role": "user", "content": [{"type": "text", "text": "Who are you?"}]}]

pipe = pipeline("image-text-to-text", model=model_path, device=0, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=64)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForMultimodalLM

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForMultimodalLM.from_pretrained(model_path).to("cuda").eval()

messages = [{'role': 'user',
  'content': [{'type': 'image',
               'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
              {'type': 'text', 'text': 'What animal is on the candy?'}]}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
